# RUKOPYS Phase 3 + Phase 4 Kaggle Runner

This notebook is built for the artifact layout from:

- `bnthanh/htr-01-train-detector-output`
- `ngovietan/htr-02-train-recognizer`
- raw RUKOPYS dataset, e.g. `bnthanh/rukopys-dataset`

Design goals:

- run Phase 3/4 under an 11 hour Kaggle budget;
- use QLoRA 4-bit recognizer safely on Kaggle T4x2;
- avoid re-training;
- write resumable progress every few test images;
- always leave a valid `submission.csv`, even if the run stops early.


In [ ]:
# ============================================================
# 0. User knobs
# ============================================================
# Resume mode for finishing Phase 4 from a previous Phase 3/4 output dataset.
# Attach these Kaggle inputs:
#   - bnthanh/rukopys-dataset
#   - bnthanh/htr-01-train-detector-output
#   - ngovietan/htr-02-train-recognizer
#   - your previous 03_04 output dataset containing phase4_progress.jsonl
RUN_PHASE3 = False                # skip quick grid; reuse previous best_config/threshold
RUN_PHASE4 = True                 # continue submission generation
TIME_BUDGET_HOURS = 11.0          # stop safely before Kaggle 12h hard limit

# Phase 3 knobs are kept only if you intentionally set RUN_PHASE3=True.
PHASE3_N_VALID = 8
CONF_GRID = "0.05,0.10,0.20"
IOU_GRID = "0.45"

# Phase 4 checkpointing/resume knobs.
SAVE_EVERY_IMAGES = 1             # save every image; output is tiny and timeout-safe
MAX_TEST_IMAGES = 0               # 0 = all test images; resume will skip completed ones

# T4x2-safe inference settings for QLoRA.
OCR_BATCH = 1
MAX_TOKENS = 96                   # faster/safer than 128 for resume; raise if long text is cut
MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 384 * 28 * 28        # lower = faster/safer; 512*28*28 if you have time
YOLO_IMGSZ = 1024                 # detector was trained at 1024 in the trial run
USE_TTA = False

# If Phase 3 is skipped or fails, these thresholds are used.
DEFAULT_YOLO_CONF = 0.20
DEFAULT_YOLO_IOU = 0.45


In [ ]:

# ============================================================
# 1. Setup repo, env, dependencies
# ============================================================
import os, sys, json, time, shutil, subprocess, gc, tarfile
from pathlib import Path

ROOT = Path('/kaggle/working/rukopys')
REPO_URL = 'https://github.com/thanhbinh55/rukopys.git'


def run(cmd, check=True):
    print(f"\n$ {cmd}")
    return subprocess.run(cmd, shell=True, executable='/bin/bash', check=check)

if not ROOT.exists():
    run(f'git clone {REPO_URL} {ROOT}')
elif not (ROOT / 'scripts' / 'inference_utils.py').exists():
    shutil.rmtree(ROOT)
    run(f'git clone {REPO_URL} {ROOT}')
else:
    run(f'git -C {ROOT} fetch origin main', check=False)
    run(f'git -C {ROOT} reset --hard origin/main', check=False)

# Minimal runtime deps. Keep installs focused to reduce startup time.
run(f"""
{sys.executable} -m pip install -q -U pip wheel setuptools packaging
{sys.executable} -m pip install -q \
  'transformers==4.57.1' \
  'peft==0.17.1' \
  'accelerate>=1.0.0,<2.0.0' \
  'huggingface_hub>=0.24.0' \
  'tokenizers>=0.20.0' \
  qwen-vl-utils timm bitsandbytes ultralytics \
  pandas numpy pillow rapidfuzz pyyaml tensorboard
""")

os.chdir(ROOT)
print('ROOT =', ROOT)



In [ ]:

# ============================================================
# 2. Discover Kaggle input paths and normalize artifacts
# ============================================================
from pathlib import Path
import os, json, shutil

KINPUT = Path('/kaggle/input')
ART = ROOT / 'local_working' / 'htr_artifacts'
ART.mkdir(parents=True, exist_ok=True)
(ART / 'logs').mkdir(parents=True, exist_ok=True)


def has_raw_dataset(p: Path) -> bool:
    return (p / 'train' / 'metadata.jsonl').exists() and (p / 'test' / 'metadata.jsonl').exists()


def find_raw_data_root() -> Path:
    candidates = [
        Path('/kaggle/input/datasets/bnthanh/rukopys-dataset/rukopys_raw'),
        Path('/kaggle/input/datasets/bnthanh/rukopys-dataset'),
        KINPUT / 'rukopys-dataset' / 'rukopys_raw',
        KINPUT / 'rukopys-dataset',
    ]
    for p in candidates:
        if has_raw_dataset(p):
            return p
    for p in KINPUT.glob('**/rukopys_raw'):
        if has_raw_dataset(p):
            return p
    raise FileNotFoundError('Could not find raw RUKOPYS dataset. Attach bnthanh/rukopys-dataset.')


def first_existing(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


def copy_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'copied {src} -> {dst}')


def copy_dir(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'copied dir {src} -> {dst}')

DATA_ROOT = find_raw_data_root()
print('DATA_ROOT =', DATA_ROOT)

# Detector artifact dataset.
DET_ROOTS = [
    Path('/kaggle/input/datasets/bnthanh/htr-01-train-detector-output'),
    KINPUT / 'htr-01-train-detector-output',
]
REC_ROOTS = [
    Path('/kaggle/input/datasets/ngovietan/htr-02-train-recognizer'),
    KINPUT / 'htr-02-train-recognizer',
]

DET_ROOT = next((p for p in DET_ROOTS if p.exists()), None)
REC_ROOT = next((p for p in REC_ROOTS if p.exists()), None)
print('DET_ROOT =', DET_ROOT)
print('REC_ROOT =', REC_ROOT)
if DET_ROOT is None:
    raise FileNotFoundError('Attach detector output dataset: bnthanh/htr-01-train-detector-output')
if REC_ROOT is None:
    raise FileNotFoundError('Attach recognizer output dataset: ngovietan/htr-02-train-recognizer')

# Copy detector weights only, not the whole 10GB dataset.
det_yolo_dst = ART / 'det_yolo'
det_yolo_dst.mkdir(parents=True, exist_ok=True)
for name in ['best.pt', 'last.pt', 'best.engine', 'data.yaml']:
    hits = list(DET_ROOT.glob(f'**/det_yolo/{name}')) + list(DET_ROOT.glob(f'**/{name}'))
    if hits:
        copy_file(hits[0], det_yolo_dst / name)

if not (det_yolo_dst / 'best.pt').exists():
    raise FileNotFoundError('det_yolo/best.pt not found in detector output dataset')

# Copy recognizer LoRA best checkpoint. This is small enough and needed for inference.
qwen_dst = ART / 'qwen3_lora'
qwen_dst.mkdir(parents=True, exist_ok=True)
best_hits = list(REC_ROOT.glob('**/qwen3_lora/best_checkpoint')) + list(REC_ROOT.glob('**/best_checkpoint'))
if best_hits:
    copy_dir(best_hits[0], qwen_dst / 'best_checkpoint')
else:
    ckpts = sorted(
        REC_ROOT.glob('**/qwen3_lora/checkpoint-*'),
        key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1,
    )
    if not ckpts:
        ckpts = sorted(
            REC_ROOT.glob('**/checkpoint-*'),
            key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1,
        )
    if not ckpts:
        raise FileNotFoundError('No best_checkpoint or checkpoint-* found in recognizer dataset')
    copy_dir(ckpts[-1], qwen_dst / 'best_checkpoint')

# Copy Phase 0 artifacts from recognizer output if available; otherwise regenerate.
for name in ['config.json', 'train_split.jsonl', 'valid_split.jsonl', 'valid_gt.csv', 'scorable_stats.json']:
    hits = list(REC_ROOT.glob(f'**/htr_artifacts/{name}')) + list(REC_ROOT.glob(f'**/{name}'))
    if hits:
        copy_file(hits[0], ART / name)


# Resume from a previous Phase 3/4 output dataset if one is attached.
# We pick the phase4_progress.jsonl with the most rows under /kaggle/input.
def count_lines(path: Path) -> int:
    try:
        with open(path, encoding='utf-8') as f:
            return sum(1 for line in f if line.strip())
    except Exception:
        return 0

resume_hits = sorted(
    KINPUT.glob('**/phase4_progress.jsonl'),
    key=lambda p: count_lines(p),
    reverse=True,
)
if resume_hits:
    resume_progress = resume_hits[0]
    n_resume = count_lines(resume_progress)
    copy_file(resume_progress, ART / 'phase4_progress.jsonl')
    print(f'resume source = {resume_progress} ({n_resume} completed images)')

    resume_root = resume_progress.parent
    for name in ['best_config.json', 'grid_search_results.csv', 'phase3_4_summary.json']:
        src_file = resume_root / name
        if src_file.exists():
            copy_file(src_file, ART / name)

    # Copy previous partial CSVs into /kaggle/working for visibility; Phase 4 will rewrite them.
    for name in ['submission.csv', 'submission_partial.csv']:
        src_file = resume_root / name
        if src_file.exists():
            copy_file(src_file, Path('/kaggle/working') / name)
else:
    print('No previous phase4_progress.jsonl found under /kaggle/input; Phase 4 will start from scratch.')

# Ensure config points at the currently attached raw dataset, not the old notebook path.
need_phase0 = not (ART / 'config.json').exists() or not (ART / 'valid_split.jsonl').exists()
os.environ.update({
    'PROJECT_ROOT': str(ROOT),
    'DATA_ROOT': str(DATA_ROOT),
    'RUKOPYS_ROOT': str(DATA_ROOT),
    'HTR_ART_DIR': str(ART),
    'PYTHONPATH': f'{ROOT}:{ROOT / "scripts"}',
})
if need_phase0:
    run('python scripts/00_setup_inspect.py')
else:
    cfg = json.load(open(ART / 'config.json', encoding='utf-8'))
    cfg.update({
        'DATA_ROOT': str(DATA_ROOT),
        'TRAIN_IMG': str(DATA_ROOT / 'train' / 'images'),
        'TEST_IMG': str(DATA_ROOT / 'test' / 'images'),
        'TRAIN_META': str(DATA_ROOT / 'train' / 'metadata.jsonl'),
        'TEST_META': str(DATA_ROOT / 'test' / 'metadata.jsonl'),
        'SILVER_META': str(DATA_ROOT / 'silver' / 'metadata.jsonl'),
        'SILVER_IMG': str(DATA_ROOT / 'silver' / 'images'),
        'ART': str(ART),
    })
    json.dump(cfg, open(ART / 'config.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
    print('updated config.json paths')

print('\nArtifact summary:')
for p in [det_yolo_dst / 'best.pt', det_yolo_dst / 'best.engine', qwen_dst / 'best_checkpoint' / 'adapter_model.safetensors', ART / 'config.json', ART / 'valid_split.jsonl']:
    print(p, 'OK' if p.exists() else 'missing')



In [ ]:

# ============================================================
# 3. Runtime env for safe T4x2 QLoRA inference
# ============================================================
import os, sys, json, time, gc
from pathlib import Path

os.environ.update({
    'CUDA_VISIBLE_DEVICES': '0,1',
    'PROJECT_ROOT': str(ROOT),
    'DATA_ROOT': str(DATA_ROOT),
    'RUKOPYS_ROOT': str(DATA_ROOT),
    'HTR_ART_DIR': str(ART),
    'HF_HOME': '/tmp/hf_cache',
    'TRANSFORMERS_CACHE': '/tmp/hf_cache',
    'YOLO_CONFIG_DIR': str(ROOT / 'local_working' / 'ultralytics'),
    'PYTHONPATH': f'{ROOT}:{ROOT / "scripts"}',
    'TOKENIZERS_PARALLELISM': 'false',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True,max_split_size_mb:128',
    'USE_FLASH_ATTN': '0',
    'USE_BF16': '0',
    'USE_TF32': '0',
    'QLORA_4BIT': '1',
    'DEVICE_MAP': 'auto',
    'GPU_MAX_MEMORY': '13GiB',
    'CPU_MAX_MEMORY': '24GiB',
    'OCR_BATCH': str(OCR_BATCH),
    'MAX_TOKENS': str(MAX_TOKENS),
    'MIN_PIXELS': str(MIN_PIXELS),
    'MAX_PIXELS': str(MAX_PIXELS),
    'YOLO_IMGSZ': str(YOLO_IMGSZ),
    'USE_TTA': '1' if USE_TTA else '0',
    'YOLO_CONF': str(DEFAULT_YOLO_CONF),
    'YOLO_IOU': str(DEFAULT_YOLO_IOU),
})
Path('/tmp/hf_cache').mkdir(parents=True, exist_ok=True)

run('python - <<\'PY\'\nimport torch, transformers, bitsandbytes\nprint("torch", torch.__version__)\nprint("cuda", torch.cuda.is_available(), torch.cuda.device_count())\nfor i in range(torch.cuda.device_count()):\n    print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/1024**3, 2), "GiB")\nprint("transformers", transformers.__version__)\nprint("bitsandbytes ok")\nPY')



In [ ]:

# ============================================================
# 4. Load patched inference helpers: no TensorRT export, QLoRA recognizer
# ============================================================
import logging, math
import pandas as pd
import torch
from PIL import Image

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[logging.FileHandler(ART / 'logs' / 'phase3_4_runner.log'), logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger('phase3_4')

# Patch Transformers 4.57 generation compatibility before importing inference_utils.
# Qwen3VL generate() rejects enable_thinking as a model kwarg in this runtime.
inf_path = ROOT / 'scripts' / 'inference_utils.py'
inf_src = inf_path.read_text()
inf_src = inf_src.replace('                enable_thinking=False,\n', '')
inf_path.write_text(inf_src)

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))
import inference_utils as iu
from kaggle_metric import score_detailed

# Keep helper globals aligned with this notebook env.
iu.DEVICE = 'cuda:0'
iu.MAX_NEW_TOKENS = MAX_TOKENS


def load_yolo_detector_no_export(art=ART):
    from ultralytics import YOLO
    best_pt = Path(art) / 'det_yolo' / 'best.pt'
    engine = Path(art) / 'det_yolo' / 'best.engine'
    if engine.exists() and os.getenv('USE_TRT_ENGINE', '0') == '1':
        try:
            log.info('[yolo] loading TensorRT engine: %s', engine)
            return YOLO(str(engine))
        except Exception as e:
            log.warning('[yolo] TensorRT failed, fallback to best.pt: %s', e)
    if not best_pt.exists():
        raise FileNotFoundError(best_pt)
    log.info('[yolo] loading PyTorch detector: %s', best_pt)
    return YOLO(str(best_pt))


def yolo_detect_page_fast(img_path, detector, conf, iou, use_tta=False):
    results = detector.predict(
        source=str(img_path), conf=float(conf), iou=float(iou), imgsz=int(YOLO_IMGSZ),
        device='cuda:0', augment=bool(use_tta), verbose=False,
    )
    if not results or not results[0].boxes:
        return []
    regions = []
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls_id = int(box.cls[0].item())
        regions.append({
            'bbox': [int(x1), int(y1), int(x2), int(y2)],
            'type': iu.CLASSES[cls_id] if cls_id < len(iu.CLASSES) else 'handwritten',
            'conf': float(box.conf[0].item()),
            'text': '',
        })
    return regions


def load_qwen3_model_qlora(art=ART):
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
    from peft import PeftModel

    art = Path(art)
    lora_dir = art / 'qwen3_lora' / 'best_checkpoint'
    if not lora_dir.exists():
        ckpts = sorted((art / 'qwen3_lora').glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
        if not ckpts:
            raise FileNotFoundError(f'No LoRA checkpoint under {art / "qwen3_lora"}')
        lora_dir = ckpts[-1]
    model_id = os.getenv('MODEL_ID', 'Qwen/Qwen3-VL-8B-Instruct')
    dtype = torch.float16

    log.info('[ocr] base=%s lora=%s', model_id, lora_dir)
    processor = AutoProcessor.from_pretrained(
        str(lora_dir),
        min_pixels=int(os.getenv('MIN_PIXELS', str(MIN_PIXELS))),
        max_pixels=int(os.getenv('MAX_PIXELS', str(MAX_PIXELS))),
    )
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )
    max_memory = {i: os.getenv('GPU_MAX_MEMORY', '13GiB') for i in range(torch.cuda.device_count())}
    max_memory['cpu'] = os.getenv('CPU_MAX_MEMORY', '24GiB')
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=dtype,
        attn_implementation='sdpa',
        quantization_config=quant,
        device_map='auto',
        max_memory=max_memory,
    )
    model = PeftModel.from_pretrained(model, str(lora_dir))
    model.eval()
    if hasattr(model, 'generation_config') and hasattr(model.generation_config, 'enable_thinking'):
        model.generation_config.enable_thinking = False
    log.info('[ocr] loaded QLoRA. GPU0 allocated %.2fGB', torch.cuda.memory_allocated(0) / 1e9)
    return model, processor


def write_ordered_submission(rows_by_image, output_csv, test_rows=None):
    output_csv = Path(output_csv)
    sample_path = DATA_ROOT / 'sample_submission.csv'
    ordered = []
    if sample_path.exists():
        sample = pd.read_csv(sample_path)
        for image in sample['image'].astype(str):
            ordered.append({'image': image, 'regions': rows_by_image.get(image, '[]')})
    elif test_rows is not None:
        for meta in test_rows:
            name = Path(str(meta.get('image') or meta.get('file_name') or meta.get('id') or '')).name
            ordered.append({'image': name, 'regions': rows_by_image.get(name, '[]')})
    else:
        ordered = [{'image': k, 'regions': v} for k, v in sorted(rows_by_image.items())]
    df = pd.DataFrame(ordered)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    return df


def read_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def metadata_image_name(row):
    return str(row.get('image') or row.get('file_name') or row.get('id') or '')

print('helpers ready')




In [ ]:

# ============================================================
# 5. Phase 3: quick validation and threshold selection
# ============================================================
start_time = time.time()
summary = {
    'run_phase3': RUN_PHASE3,
    'run_phase4': RUN_PHASE4,
    'time_budget_hours': TIME_BUDGET_HOURS,
    'phase3': None,
    'phase4': None,
}

cfg = json.load(open(ART / 'config.json', encoding='utf-8'))
train_img_index = iu.build_image_index(Path(cfg['TRAIN_IMG']))
valid_rows_all = read_jsonl(ART / 'valid_split.jsonl')

if RUN_PHASE3:
    log.info('PHASE 3 quick validation: N=%d conf=%s iou=%s', PHASE3_N_VALID, CONF_GRID, IOU_GRID)
    detector = load_yolo_detector_no_export(ART)
    ocr_model, ocr_processor = load_qwen3_model_qlora(ART)
    valid_rows = valid_rows_all[:PHASE3_N_VALID]
    gt_rows = []
    for r in valid_rows:
        img_name = metadata_image_name(r)
        gt_rows.append({'image': img_name, 'regions': json.dumps(r.get('regions') or r.get('annotations') or [], ensure_ascii=False)})
    gt_df = pd.DataFrame(gt_rows)

    results = []
    best = {'score': -1, 'conf': DEFAULT_YOLO_CONF, 'iou': DEFAULT_YOLO_IOU}
    for conf in [float(x) for x in CONF_GRID.split(',') if x.strip()]:
        for iou in [float(x) for x in IOU_GRID.split(',') if x.strip()]:
            rows = []
            bs = OCR_BATCH
            t0 = time.time()
            for k, meta in enumerate(valid_rows, start=1):
                img_name = metadata_image_name(meta)
                img_path = iu.resolve_image_path(img_name, train_img_index)
                if img_path is None:
                    rows.append({'image': img_name, 'regions': '[]'})
                    continue
                regs = yolo_detect_page_fast(img_path, detector, conf, iou, use_tta=USE_TTA)
                regs = iu.reading_order_sort(regs)
                regs, bs = iu.ocr_regions(str(img_path), regs, ocr_model, ocr_processor, bs)
                rows.append({'image': img_name, 'regions': iu.regions_to_submission_json(regs)})
                if k % 2 == 0:
                    log.info('[phase3 conf=%.2f iou=%.2f] %d/%d', conf, iou, k, len(valid_rows))
            pred_df = pd.DataFrame(rows)
            score = score_detailed(gt_df, pred_df, 'image')
            row = {
                'conf': conf, 'iou': iou,
                'composite': score['composite_score'],
                'det_f1': score['detection_f1'],
                'region_cer': score['region_cer'],
                'page_cer': score['page_cer'],
                'elapsed_sec': round(time.time() - t0, 1),
            }
            results.append(row)
            log.info('[phase3] %s', row)
            if row['composite'] > best['score']:
                best = {'score': row['composite'], 'conf': conf, 'iou': iou}
            pd.DataFrame(results).to_csv(ART / 'grid_search_results.csv', index=False)
            torch.cuda.empty_cache(); gc.collect()

    best_config = {
        'conf': best['conf'],
        'iou': best['iou'],
        'best_composite_score': best['score'],
        'n_valid_images': PHASE3_N_VALID,
        'quick_grid': True,
        'use_tta': USE_TTA,
    }
    json.dump(best_config, open(ART / 'best_config.json', 'w', encoding='utf-8'), indent=2)
    summary['phase3'] = best_config
    log.info('PHASE 3 best_config=%s', best_config)
    del ocr_model, ocr_processor, detector
    torch.cuda.empty_cache(); gc.collect()
else:
    best_config = {'conf': DEFAULT_YOLO_CONF, 'iou': DEFAULT_YOLO_IOU, 'quick_grid': False}
    json.dump(best_config, open(ART / 'best_config.json', 'w', encoding='utf-8'), indent=2)
    summary['phase3'] = best_config
    log.info('PHASE 3 skipped. Using defaults: %s', best_config)



In [ ]:

# ============================================================
# 6. Phase 4: resumable inference with periodic progress checkpoints
# ============================================================
if RUN_PHASE4:
    log.info('PHASE 4 inference start')
    cfg = json.load(open(ART / 'config.json', encoding='utf-8'))
    test_meta = Path(cfg.get('TEST_META', DATA_ROOT / 'test' / 'metadata.jsonl'))
    test_img = Path(cfg.get('TEST_IMG', DATA_ROOT / 'test' / 'images'))
    test_rows = read_jsonl(test_meta)
    if MAX_TEST_IMAGES and MAX_TEST_IMAGES > 0:
        test_rows = test_rows[:MAX_TEST_IMAGES]
        log.info('MAX_TEST_IMAGES active: %d', len(test_rows))
    test_img_index = iu.build_image_index(test_img)

    best_cfg = json.load(open(ART / 'best_config.json', encoding='utf-8')) if (ART / 'best_config.json').exists() else {}
    conf = float(best_cfg.get('conf', DEFAULT_YOLO_CONF))
    iou = float(best_cfg.get('iou', DEFAULT_YOLO_IOU))
    log.info('thresholds conf=%.3f iou=%.3f', conf, iou)

    progress_jsonl = ART / 'phase4_progress.jsonl'
    partial_csv = Path('/kaggle/working/submission_partial.csv')
    final_csv = Path('/kaggle/working/submission.csv')
    rows_by_image = {}
    if progress_jsonl.exists():
        with open(progress_jsonl, encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    rows_by_image[r['image']] = r['regions']
        log.info('resumed %d images from %s', len(rows_by_image), progress_jsonl)

    detector = load_yolo_detector_no_export(ART)
    ocr_model, ocr_processor = load_qwen3_model_qlora(ART)
    bs = OCR_BATCH
    phase4_start = time.time()
    processed_this_run = 0

    with open(progress_jsonl, 'a', encoding='utf-8') as pf:
        for idx, meta in enumerate(test_rows, start=1):
            elapsed_total = (time.time() - start_time) / 3600
            if elapsed_total >= TIME_BUDGET_HOURS:
                log.warning('time budget %.2fh reached at %d/%d; saving partial submission', elapsed_total, idx, len(test_rows))
                break

            image_name = metadata_image_name(meta)
            submit_name = Path(image_name).name
            if submit_name in rows_by_image:
                continue

            img_path = iu.resolve_image_path(image_name, test_img_index)
            if img_path is None:
                log.warning('missing test image: %s', image_name)
                regions_json = '[]'
            else:
                regs = yolo_detect_page_fast(img_path, detector, conf, iou, use_tta=USE_TTA)
                if regs:
                    regs = iu.reading_order_sort(regs)
                    regs, bs = iu.ocr_regions(str(img_path), regs, ocr_model, ocr_processor, bs)
                regions_json = iu.regions_to_submission_json(regs)

            rows_by_image[submit_name] = regions_json
            pf.write(json.dumps({'image': submit_name, 'regions': regions_json}, ensure_ascii=False) + '\n')
            pf.flush()
            processed_this_run += 1

            if processed_this_run % SAVE_EVERY_IMAGES == 0 or idx == len(test_rows):
                df = write_ordered_submission(rows_by_image, partial_csv, test_rows=test_rows)
                shutil.copy2(partial_csv, final_csv)
                log.info('[phase4] saved %d/%d images -> %s', len(rows_by_image), len(test_rows), final_csv)
                torch.cuda.empty_cache(); gc.collect()

    # Always write a valid submission; missing images are filled with [] by write_ordered_submission.
    df = write_ordered_submission(rows_by_image, final_csv, test_rows=test_rows)
    shutil.copy2(final_csv, ART / 'submission.csv')
    summary['phase4'] = {
        'processed_images': len(rows_by_image),
        'target_images': len(test_rows),
        'complete': len(rows_by_image) >= len(test_rows),
        'submission_csv': str(final_csv),
        'elapsed_sec': round(time.time() - phase4_start, 1),
    }
    log.info('PHASE 4 summary: %s', summary['phase4'])

    del ocr_model, ocr_processor, detector
    torch.cuda.empty_cache(); gc.collect()
else:
    log.info('PHASE 4 skipped')



In [ ]:

# ============================================================
# 7. Final packaging: small, useful output only
# ============================================================
summary_path = Path('/kaggle/working/phase3_4_summary.json')
json.dump(summary, open(summary_path, 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2, ensure_ascii=False))

bundle = Path('/kaggle/working/phase3_4_artifacts.tgz')
items = [
    Path('/kaggle/working/submission.csv'),
    Path('/kaggle/working/submission_partial.csv'),
    summary_path,
    ART / 'best_config.json',
    ART / 'grid_search_results.csv',
    ART / 'phase4_progress.jsonl',
    ART / 'logs' / 'phase3_4_runner.log',
]
with tarfile.open(bundle, 'w:gz') as tar:
    for item in items:
        if item.exists():
            tar.add(item, arcname=item.name)
print('Wrote bundle:', bundle)
run('ls -lh /kaggle/working/submission.csv /kaggle/working/phase3_4_artifacts.tgz 2>/dev/null || true')
run('df -h /kaggle/working /tmp')

